# Context: dane aplikacji przekazywane do agenta

W poprzedniej lekcji `state` przechowywał komunikaty i wynik agenta. Tutaj aplikacja zna dodatkowe dane o **bieżącym wywołaniu**: identyfikator użytkownika i preferowany język. Przekażemy je przez `context=...`, poza treścią pytania. Przykład logowania i baza danych są symulowane: nie ma hasła, serwera, połączenia SQL ani operacji na koncie.

Na końcu zobaczymy dwa zastosowania tych samych danych: kod aplikacji przygotuje plan zapytania do bazy, a narzędzie agenta odczyta profil z lokalnego słownika. Model zobaczy tylko te informacje, które celowo trafią do promptu lub wyniku narzędzia.

[Runtime context w LangChain](https://docs.langchain.com/oss/python/langchain/tools#context) · [Agent invocation](https://docs.langchain.com/oss/python/langchain/agents#invocation)


In [1]:
# Wszystkie importy i minimalne ustawienie ścieżki projektu są w tej komórce.
import sys
from dataclasses import dataclass
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from langchain.agents import create_agent
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain.tools import ToolRuntime, tool
from model_config import get_model


## 1. Skąd bierze się `user_id`?

W prawdziwej aplikacji serwer ustala tożsamość po sprawdzeniu sesji/logowania i dopiero wtedy tworzy kontekst. Poniższe słowniki **udają wynik tego kroku**. Nie są mechanizmem logowania. Nie bierzemy `user_id` z pytania użytkownika ani z argumentu wymyślonego przez model.

`RequestContext` opisuje dane dostępne podczas jednego wywołania. Zmieniamy kontekst, gdy obsługujemy innego użytkownika; nie musimy zmieniać tekstu pytania ani tworzyć nowego agenta.


In [2]:
@dataclass(frozen=True)
class RequestContext:
    user_id: str
    language: str


# Fikcyjne wyniki wcześniej zweryfikowanych sesji; tutaj niczego nie uwierzytelniamy.
MOCK_SESSIONS = [
    {"subject": "u-101", "language": "pl"},
    {"subject": "u-202", "language": "en"},
]
contexts = [
    RequestContext(user_id=session["subject"], language=session["language"])
    for session in MOCK_SESSIONS
]
print("Konteksty utworzone przez aplikację:", contexts)


Konteksty utworzone przez aplikację: [RequestContext(user_id='u-101', language='pl'), RequestContext(user_id='u-202', language='en')]


## 2. Ten sam kontekst może wykorzystać kod aplikacji

Funkcja poniżej **tylko pokazuje plan** zapytania. `user_id` jest osobnym parametrem, a nie częścią sklejonego tekstu SQL. Nie ma połączenia z bazą ani `execute()`. W działającym systemie uprawnienia do odczytu musiałaby egzekwować warstwa aplikacji/bazy.


In [3]:
def preview_profile_query(context: RequestContext) -> dict:
    return {
        "sql": "SELECT display_name, plan FROM profiles WHERE user_id = ?",
        "params": (context.user_id,),
    }


print("Plan dla pierwszej sesji:", preview_profile_query(contexts[0]))
print("Plan dla drugiej sesji:", preview_profile_query(contexts[1]))


Plan dla pierwszej sesji: {'sql': 'SELECT display_name, plan FROM profiles WHERE user_id = ?', 'params': ('u-101',)}
Plan dla drugiej sesji: {'sql': 'SELECT display_name, plan FROM profiles WHERE user_id = ?', 'params': ('u-202',)}


## 3. Narzędzie czyta `runtime.context`

Zamiast prawdziwej bazy używamy dwóch fikcyjnych rekordów w słowniku. Narzędzie `my_profile` **nie ma argumentu `user_id` w schemacie widocznym dla modelu**. Gdy agent je wywoła, LangChain wstrzyknie `ToolRuntime`; kod narzędzia weźmie ID z kontekstu dostarczonego przez aplikację. Model zobaczy dopiero zwrócone imię i plan, bez surowego ID.


In [5]:
DEMO_PROFILES = {
    "u-101": {"display_name": "Ala", "plan": "Plus"},
    "u-202": {"display_name": "Bartek", "plan": "Basic"},
}


@tool
def my_profile(runtime: ToolRuntime[RequestContext]) -> str:

    """Odczytaj nazwę i plan bieżącego użytkownika z danych demonstracyjnych."""
    profile = DEMO_PROFILES.get(runtime.context.user_id)

    if profile is None:
        return "Brak profilu w danych demonstracyjnych."
    return f"Imię: {profile['display_name']}; plan: {profile['plan']}"


## 4. Co z kontekstu pokazujemy modelowi?

Samo `context=...` nie dokleja danych do promptu. Tu `dynamic_prompt` świadomie przekazuje **tylko preferowany język**. ID pozostaje w kodzie aplikacji i narzędzia. Prompt zapisujemy do listy wyłącznie po to, by było widać, co naprawdę dostał model. Ta lista również działa tylko w bieżącym kernelu.


In [14]:
PROMPTS_SHOWN_TO_MODEL = []


@dynamic_prompt
def language_prompt(request: ModelRequest) -> str:
    language = "po polsku" if request.runtime.context.language == "pl" else "po angielsku"
    base_prompt = request.system_message.text if request.system_message else ""
    prompt = f"{base_prompt}\nOdpowiadaj {language}."
    PROMPTS_SHOWN_TO_MODEL.append(prompt)
    return prompt


model = get_model()
agent = create_agent(
    model=model,
    tools=[my_profile],
    context_schema=RequestContext,
    system_prompt="Gdy użytkownik pyta o swój plan, sprawdź dane, nigdy ich nie zgaduj. Oczywiście odpowiedz w języku usera",
    middleware=[language_prompt],
)


MODEL [main] profile=local provider=ollama model=gemma4:12b


## 5. To samo pytanie, dwa różne konteksty

Pytanie nie zawiera ID ani nazwy planu. Każde `agent.invoke(...)` dostaje osobny `context`. Agent może poprosić o `my_profile`, Python odczytuje właściwy rekord demonstracyjny, a model formułuje odpowiedź na podstawie wyniku narzędzia. Wyświetlamy komunikaty narzędzia osobno, żeby nie mieszać decyzji modelu z tym, co zwrócił Python.


In [15]:
QUESTION = "Jaki mam plan? Odpowiedz krótko."


def ask_about_plan(context: RequestContext) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": QUESTION}]},
        context=context,
    )
    print("Kontekst aplikacji:", context)
    for message in result["messages"]:
        if message.type == "ai" and message.tool_calls:
            print("Model poprosił o:", [call["name"] for call in message.tool_calls])
        elif message.type == "tool":
            print("Python zwrócił z narzędzia:", message.content)
    print("Odpowiedź modelu:", result["messages"][-1].content, "\n\n")
    return result


first_result = ask_about_plan(contexts[0])
second_result = ask_about_plan(contexts[1])



Kontekst aplikacji: RequestContext(user_id='u-101', language='pl')
Model poprosił o: ['my_profile']
Python zwrócił z narzędzia: Imię: Ala; plan: Plus
Odpowiedź modelu: Twój plan to Plus. 


Kontekst aplikacji: RequestContext(user_id='u-202', language='en')
Model poprosił o: ['my_profile']
Python zwrócił z narzędzia: Imię: Bartek; plan: Basic
Odpowiedź modelu: Your plan is Basic. 




## Co właśnie się wydarzyło?

| Dane | Kto je ustala? | Kto je widzi w tym przykładzie? |
|---|---|---|
| Pytanie `Jaki mam plan?` | użytkownik | model |
| `user_id` | fikcyjna sesja aplikacji | kod aplikacji i narzędzie przez `runtime.context` |
| `language` | fikcyjna sesja aplikacji | kod aplikacji; model dopiero po celowym dodaniu do systemowego promptu |
| Imię i plan | lokalny słownik udający odczyt | narzędzie, a potem model przez wynik narzędzia |

`context` to dane bieżącego wywołania dla narzędzi i middleware. `state` z poprzedniej lekcji to dane przebiegu agenta, np. `messages` i `structured_response`. `thread_id` z lekcji o memory wskazuje rozmowę zapisywaną przez checkpointer. W tym notebooku nie ma checkpointera, prawdziwego logowania ani połączenia z bazą.

Kluczowa granica: model może **poprosić** o narzędzie i opisać wynik, ale nie wybiera, czyim ID narzędzie ma się posłużyć. W prawdziwym systemie aplikacja musi zweryfikować sesję i uprawnienia przed przekazaniem kontekstu.
